In [1]:
# =============================================================================
# CS-781 Phase 2 — L-NER, Experiment 1: FULL FINE-TUNING of Arch #1 (decoder-only)
# BIO token-classification head on top of the Phase-1 backbone.
#
# Key design points (all about strict character-offset correctness):
#   * Words are found with a regex over the ORIGINAL text, so every BPE token
#     keeps an exact character range (the Phase-1 encode() split on whitespace
#     and lost offsets).
#   * Gold char spans -> token BIO labels, then an ORACLE CHECK measures how many
#     gold spans can be recovered exactly from token boundaries, and picks the
#     best leading/trailing punctuation trim rule from the training data.
#   * Long documents are cut into overlapping windows (<= block_size); for each
#     token we keep the prediction from the window where it has the most left
#     context (the backbone is causal, so left context is all it sees).
#
# Run in Kaggle: paste each "# %%" block as its own cell, in order.
# =============================================================================

In [2]:

# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, itertools
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda


In [3]:
# %% [2] Config
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

# Tokenizer artifacts (same dataset used in Phase 1)
ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/phase1token"
# !!! EDIT: the Arch #1 RETRAIN checkpoint (model_best.pt) attached as a Kaggle input
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_TEXT_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold2-text.json"]
TRAIN_GOLD_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold2-gold.json"]
LABELS_FILE = "Phase_2_LNER/Train/labels.txt"
TEST_TEXT_FILE = "Phase_2_LNER/Test/fold3-text.json"

VAL_FRACTION_DOCS = 0.15      # held-out docs for local strict-F1
STRIDE = 512                  # window stride (tokens)
EPOCHS = 6
BATCH_SIZE = 8
EVAL_BATCH = 8
LR_BACKBONE = 5e-5
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1
O_WEIGHT = 1.0                # class weight for the "O" tag (try 0.3-0.5 later as a class-balancing ablation)

OUT_DIR = "/kaggle/working/cs781_phase2_exp1"
os.makedirs(OUT_DIR, exist_ok=True)
BEST_PATH = os.path.join(OUT_DIR, "ner_best.pt")
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = os.path.join(OUT_DIR, "submission_phase2_exp1.zip")

In [4]:
# %% [3] Load Phase-2 data + inspect the gold format
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

LABELS = [x.strip() for x in open(hf_local(LABELS_FILE), encoding="utf-8") if x.strip()]
assert len(LABELS) == 12, LABELS
print("Labels:", LABELS)

# BIO tag ids: 0 = O, then B/I for each label
ID2TAG = {0: ("O", None)}
for i, lab in enumerate(LABELS):
    ID2TAG[1 + 2 * i] = ("B", lab)
    ID2TAG[2 + 2 * i] = ("I", lab)
NUM_TAGS = len(ID2TAG)
B_ID = {lab: 1 + 2 * i for i, lab in enumerate(LABELS)}
I_ID = {lab: 2 + 2 * i for i, lab in enumerate(LABELS)}

def norm_entity(e):
    if isinstance(e, (list, tuple)):
        s, en, lab = e[0], e[1], e[2]
        txt = e[3] if len(e) > 3 else None
    elif isinstance(e, dict):
        def pick(*ks):
            for k in ks:
                if k in e:
                    return e[k]
            return None
        s = pick("start", "start_char", "start_offset", "begin")
        en = pick("end", "end_char", "end_offset")
        lab = pick("label", "type", "entity", "tag")
        txt = pick("text", "entity_text", "span")
    else:
        raise ValueError(f"Unrecognised entity format: {e!r}")
    return int(s), int(en), str(lab), txt

texts, gold = {}, {}
for tf, gf in zip(TRAIN_TEXT_FILES, TRAIN_GOLD_FILES):
    t = json.load(open(hf_local(tf), encoding="utf-8"))
    g = json.load(open(hf_local(gf), encoding="utf-8"))
    print(f"{tf}: {len(t)} docs | {gf}: type={type(g).__name__}, len={len(g)}")
    for k, v in t.items():
        assert k not in texts, f"duplicate doc id {k}"
        texts[k] = v
    if not isinstance(g, dict):
        raise ValueError("Gold file is not a dict keyed by doc id — paste me its first record and I'll adapt the parser.")
    for k, v in g.items():
        if isinstance(v, dict):
            v = v.get("entities") or v.get("annotations") or v.get("labels") or []
        gold[k] = [norm_entity(e) for e in v]

missing = [k for k in texts if k not in gold]
print("Docs:", len(texts), "| docs without gold entry:", len(missing))
for k in missing:
    gold[k] = []

# Verify offset convention (end exclusive) using the entity text when provided
n_chk = n_ok = 0
for k, ents in gold.items():
    for (s, e, lab, txt) in ents:
        if txt is not None:
            n_chk += 1
            n_ok += int(texts[k][s:e] == txt)
print(f"Offset check text[s:e]==entity_text: {n_ok}/{n_chk}")
if n_chk and n_ok < 0.95 * n_chk:
    print("WARNING: offsets may not be end-exclusive or text differs — inspect before trusting results.")

bad_lab = Counter(l for ents in gold.values() for (_, _, l, _) in ents if l not in LABELS)
assert not bad_lab, f"Unexpected labels in gold: {bad_lab}"
lab_counts = Counter(l for ents in gold.values() for (_, _, l, _) in ents)
print("Entity counts per label:", dict(lab_counts))
k0 = next(iter(texts))
print("Sample doc", k0, "len", len(texts[k0]), "| first gold:", gold[k0][:3])

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold1-text.json: 35 docs | Phase_2_LNER/Train/fold1-gold.json: type=dict, len=35


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold2-text.json: 35 docs | Phase_2_LNER/Train/fold2-gold.json: type=dict, len=35
Docs: 70 | docs without gold entry: 0
Offset check text[s:e]==entity_text: 8393/8393
Entity counts per label: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}
Sample doc 115651329 len 18616 | first gold: [(137, 153, 'RESP', 'STATE OF HARYANA'), (252, 261, 'DATE', '19.8.2011'), (276, 308, 'COURT', 'High Court of Punjab and Haryana')]


In [5]:
# %% [4] Tokenizer (same BPE as Phase 1) + tokenization WITH exact char offsets
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
EOW = "\u0100"
WORD_RE = re.compile(r"\S+")

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

def tokenize_with_offsets(text):
    """Returns token ids and, per token, its (char_start, char_end) in `text`.
    Tokens that are only the end-of-word marker get a zero-length range."""
    ids, offs = [], []
    for m in WORD_RE.finditer(text):
        word, w0 = m.group(), m.start()
        b2c = []
        for ci, ch in enumerate(word):
            for _ in ch.encode("utf-8", "ignore"):
                b2c.append(ci)
        bpos = 0
        for sym in bpe_encode_word(word):
            nb = sum(1 for ch in sym if ch != EOW)
            if nb == 0 or not b2c:
                ce = w0 + len(word)
                offs.append((ce, ce))
            else:
                sb = min(bpos, len(b2c) - 1)
                eb = min(bpos + nb, len(b2c))
                offs.append((w0 + b2c[sb], w0 + b2c[eb - 1] + 1))
            bpos += nb
            ids.append(token_to_id.get(sym, UNK_ID))
    return ids, offs

# sanity: same token sequence as the Phase-1 encoder
def _phase1_encode(text):
    out = []
    for w in text.split():
        for sym in bpe_encode_word(w):
            out.append(token_to_id.get(sym, UNK_ID))
    return out
_t = texts[k0][:5000]
assert tokenize_with_offsets(_t)[0] == _phase1_encode(_t), "tokenization mismatch vs Phase 1"
print("Tokenizer OK. Vocab size:", VOCAB_SIZE)

Tokenizer OK. Vocab size: 32000


In [6]:

# %% [5] Build per-document records: token ids, offsets, BIO labels (+ oracle info)
def encode_doc(text, spans=None):
    ids, offs = tokenize_with_offsets(text)
    nz = [i for i, (a, b) in enumerate(offs) if b > a]
    rec = {"ids": ids, "offs": offs, "nz": nz, "len": len(text)}
    if spans is None:
        return rec
    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    char_sid = [-1] * len(text)
    overlaps = 0
    for k, (s, e, lab, _) in enumerate(spans):
        for c in range(max(0, s), min(len(text), e)):
            if char_sid[c] != -1:
                overlaps += 1
            char_sid[c] = k
    labels = [-100] * len(ids)
    span_rng = {}
    prev_sid = -1
    for t in nz:
        cs, ce = offs[t]
        sid = -1
        for c in range(cs, ce):
            if char_sid[c] >= 0:
                sid = char_sid[c]
                break
        if sid < 0:
            labels[t] = 0
        else:
            lab = spans[sid][2]
            labels[t] = B_ID[lab] if sid != prev_sid else I_ID[lab]
            lo, hi = span_rng.get(sid, (cs, ce))
            span_rng[sid] = (min(lo, cs), max(hi, ce))
        prev_sid = sid
    rec.update(labels=labels, spans=spans, span_rng=span_rng, overlaps=overlaps)
    return rec

t0 = time.time()
recs = {k: encode_doc(texts[k], gold[k]) for k in texts}
print(f"Encoded {len(recs)} docs in {time.time()-t0:.0f}s | total tokens: {sum(len(r['ids']) for r in recs.values()):,}")
print("Overlapping-gold char positions (nested/overlapping spans collapse in BIO):", sum(r["overlaps"] for r in recs.values()))


Encoded 70 docs in 6s | total tokens: 421,043
Overlapping-gold char positions (nested/overlapping spans collapse in BIO): 0


In [7]:
# %% [6] ORACLE CHECK: how many gold spans are exactly recoverable from token boundaries?
def trim_span(text, s, e, lead, trail):
    while s < e and (text[s].isspace() or text[s] in lead):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in trail):
        e -= 1
    return s, e

LEAD_OPTS = ["", "(\"'[\u201c\u2018"]
TRAIL_OPTS = ["", ",;", ",;:", ",;:.", ",;:.)\"'\u201d\u2019]", ",;:)\"'\u201d\u2019]"]
total_spans = sum(len(r["spans"]) for r in recs.values())
lost = sum(len(r["spans"]) - len(r["span_rng"]) for r in recs.values())
print(f"Gold spans: {total_spans} | spans with NO overlapping token (unrecoverable): {lost}")
results = []
for lead, trail in itertools.product(LEAD_OPTS, TRAIL_OPTS):
    ok = 0
    for k, r in recs.items():
        for sid, (cs, ce) in r["span_rng"].items():
            s, e = r["spans"][sid][0], r["spans"][sid][1]
            ps, pe = trim_span(texts[k], cs, ce, lead, trail)
            ok += int((ps, pe) == (s, e))
    results.append((ok / max(1, total_spans), lead, trail))
results.sort(key=lambda x: -x[0])
for rate, lead, trail in results[:6]:
    print(f"oracle exact-match {rate:.4f} | lead={lead!r} trail={trail!r}")
ORACLE_RATE, TRIM_LEAD, TRIM_TRAIL = results[0]
print(f"-> using TRIM_LEAD={TRIM_LEAD!r}, TRIM_TRAIL={TRIM_TRAIL!r}  (ceiling for strict span recall ~{ORACLE_RATE:.3f})")

Gold spans: 8393 | spans with NO overlapping token (unrecoverable): 0
oracle exact-match 0.9402 | lead='("\'[“‘' trail=',;:.'
oracle exact-match 0.9395 | lead='("\'[“‘' trail=',;:.)"\'”’]'
oracle exact-match 0.9377 | lead='' trail=',;:.'
oracle exact-match 0.9340 | lead='' trail=',;:.)"\'”’]'
oracle exact-match 0.8484 | lead='("\'[“‘' trail=',;:)"\'”’]'
oracle exact-match 0.8475 | lead='("\'[“‘' trail=',;:'
-> using TRIM_LEAD='("\'[“‘', TRIM_TRAIL=',;:.'  (ceiling for strict span recall ~0.940)


In [8]:
# %% [7] Model: Arch #1 backbone (RMSNorm + RoPE + SwiGLU) + token-classification head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        return x + self.ffn(self.ln2(x))

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

class NERModel(nn.Module):
    def __init__(self, backbone, n_embd, num_tags, dropout):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_embd, num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, idx):
        return self.classifier(self.drop(self.backbone.forward_hidden(idx)))

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
backbone = LegalLLaMA(cfg["vocab_size"], cfg["n_layer"], cfg["n_head"], cfg["n_embd"],
                      cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
backbone.load_state_dict(ckpt["model_state_dict"])
model = NERModel(backbone, cfg["n_embd"], NUM_TAGS, DROPOUT_FT).to(device)
CONTENT = cfg["block_size"] - 1          # 1 slot reserved for <s>
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Phase-1 backbone (pretrain step {ckpt.get('step')}, val_loss {ckpt.get('val_loss')}) | NER model params: {n_params/1e6:.1f}M")

Loaded Phase-1 backbone (pretrain step 5700, val_loss 1.9984889757633209) | NER model params: 66.2M


In [9]:
# %% [8] Windows, datasets, split
def make_windows(n, content, stride):
    if n <= content:
        return [(0, n)]
    starts = list(range(0, n - content + 1, stride))
    if starts[-1] + content < n:
        starts.append(n - content)
    return [(s, s + content) for s in starts]

doc_ids = sorted(recs.keys())
random.Random(SEED).shuffle(doc_ids)
n_val = max(1, int(VAL_FRACTION_DOCS * len(doc_ids)))
val_ids, train_ids = doc_ids[:n_val], doc_ids[n_val:]
print(f"Train docs: {len(train_ids)} | held-out docs: {len(val_ids)}")

class WindowDataset(Dataset):
    def __init__(self, doc_keys):
        self.items = []
        for k in doc_keys:
            r = recs[k]
            for (s, e) in make_windows(len(r["ids"]), CONTENT, STRIDE):
                x = [BOS_ID] + r["ids"][s:e]
                y = [-100] + r["labels"][s:e]
                pad = CONTENT + 1 - len(x)
                self.items.append((x + [PAD_ID] * pad, y + [-100] * pad))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        x, y = self.items[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_ds = WindowDataset(train_ids)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                          num_workers=0, pin_memory=True)
print("Training windows:", len(train_ds), "| steps/epoch:", len(train_loader))

Train docs: 60 | held-out docs: 10
Training windows: 658 | steps/epoch: 83


In [10]:
# %% [9] Inference, span decoding, strict macro-F1 (mirrors the official local evaluator)
@torch.no_grad()
def predict_token_tags(net, rec):
    net.eval()
    n = len(rec["ids"])
    wins = make_windows(n, CONTENT, STRIDE)
    store = torch.zeros(n, NUM_TAGS)
    best = torch.full((n,), -1, dtype=torch.long)
    for i in range(0, len(wins), EVAL_BATCH):
        chunk = wins[i:i + EVAL_BATCH]
        batch = torch.full((len(chunk), CONTENT + 1), PAD_ID, dtype=torch.long)
        for r_i, (s, e) in enumerate(chunk):
            batch[r_i, 0] = BOS_ID
            batch[r_i, 1:1 + e - s] = torch.tensor(rec["ids"][s:e], dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(batch.to(device))
        logits = logits.float().cpu()
        for r_i, (s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            idx = s + pos
            upd = pos > best[idx]                      # keep the window with the most left context
            store[idx[upd]] = logits[r_i, 1:1 + L][upd]
            best[idx[upd]] = pos[upd]
    return store.argmax(-1)

def decode_spans(text, tags, rec):
    spans, cur = [], None
    for t in rec["nz"]:                                 # skip zero-length (end-of-word-only) tokens
        tid = int(tags[t])
        cs, ce = rec["offs"][t]
        if tid == 0:
            if cur:
                spans.append(cur); cur = None
            continue
        tag, ent = ID2TAG[tid]
        if cur is not None and tag == "I" and cur[2] == ent:
            cur[1] = ce
        else:
            if cur:
                spans.append(cur)
            cur = [cs, ce, ent]
    if cur:
        spans.append(cur)
    out = set()
    for s, e, ent in spans:
        s2, e2 = trim_span(text, s, e, TRIM_LEAD, TRIM_TRAIL)
        if e2 > s2:
            out.add((s2, e2, ent))
    return sorted(out)

def strict_macro(truth, pred):
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in truth:
        g, p = set(truth[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}
            ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    P, R, F, per = [], [], [], {}
    for l in LABELS:
        tp, fp, fn = stats[l]
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        P.append(pr); R.append(rc); F.append(f); per[l] = (pr, rc, f)
    return sum(F) / len(F), sum(P) / len(P), sum(R) / len(R), per

def evaluate_heldout(net, keys):
    truth = {k: sorted({(s, e, l) for (s, e, l, _) in gold[k]}) for k in keys}
    pred = {k: decode_spans(texts[k], predict_token_tags(net, recs[k]), recs[k]) for k in keys}
    f1, p, r, per = strict_macro(truth, pred)
    return f1, p, r, per, truth, pred

In [11]:
# %% [10] Training loop (full fine-tuning), best epoch chosen by held-out strict macro-F1
class_w = torch.ones(NUM_TAGS)
class_w[0] = O_WEIGHT
class_w = class_w.to(device)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
body_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW(
    [{"params": body_params, "lr": LR_BACKBONE, "base": LR_BACKBONE},
     {"params": head_params, "lr": LR_HEAD, "base": LR_HEAD}],
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

total_steps = EPOCHS * len(train_loader)
warm = max(1, int(WARMUP_FRAC * total_steps))
def lr_factor(step):
    if step < warm:
        return step / warm
    prog = (step - warm) / max(1, total_steps - warm)
    return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

history, best_f1, step, t0 = [], -1.0, 0, time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, n_b = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        for g in optimizer.param_groups:
            g["lr"] = g["base"] * lr_factor(step)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(x)
        loss = F.cross_entropy(logits.float().view(-1, NUM_TAGS), y.view(-1), weight=class_w, ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item(); n_b += 1; step += 1
        if step % 50 == 0:
            print(f"  step {step}/{total_steps} | loss {run_loss/n_b:.4f} | {(time.time()-t0)/60:.1f}m")
    f1, p, r, per, _, _ = evaluate_heldout(model, val_ids)
    history.append({"epoch": epoch, "train_loss": run_loss / max(1, n_b), "val_f1": f1, "val_p": p, "val_r": r})
    print(f"[epoch {epoch}] train_loss {run_loss/max(1,n_b):.4f} | held-out strict macro-F1 {f1:.4f} (P {p:.4f}, R {r:.4f}) | {(time.time()-t0)/60:.1f}m")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": f1}, BEST_PATH)
        print("  -> new best checkpoint saved")

json.dump(history, open(os.path.join(OUT_DIR, "history.json"), "w"), indent=2)
print(f"Best held-out strict macro-F1: {best_f1:.4f}")

  step 50/498 | loss 1.2237 | 0.3m
[epoch 1] train_loss 0.8068 | held-out strict macro-F1 0.1871 (P 0.1793, R 0.2072) | 0.6m
  -> new best checkpoint saved
  step 100/498 | loss 0.1371 | 0.7m
  step 150/498 | loss 0.1129 | 1.0m
[epoch 2] train_loss 0.1094 | held-out strict macro-F1 0.2451 (P 0.2206, R 0.2828) | 1.1m
  -> new best checkpoint saved
  step 200/498 | loss 0.0816 | 1.4m
[epoch 3] train_loss 0.0784 | held-out strict macro-F1 0.2618 (P 0.2309, R 0.3117) | 1.8m
  -> new best checkpoint saved
  step 250/498 | loss 0.0598 | 1.8m
  step 300/498 | loss 0.0626 | 2.2m
[epoch 4] train_loss 0.0629 | held-out strict macro-F1 0.2701 (P 0.2374, R 0.3264) | 2.4m
  -> new best checkpoint saved
  step 350/498 | loss 0.0583 | 2.6m
  step 400/498 | loss 0.0541 | 2.9m
[epoch 5] train_loss 0.0546 | held-out strict macro-F1 0.2656 (P 0.2335, R 0.3210) | 3.1m
  step 450/498 | loss 0.0514 | 3.3m
[epoch 6] train_loss 0.0520 | held-out strict macro-F1 0.2656 (P 0.2335, R 0.3211) | 3.7m
Best held-out

In [12]:
# %% [11] Per-label breakdown of the best checkpoint (for the report's error analysis)
best = torch.load(BEST_PATH, map_location="cpu")
model.load_state_dict(best["model"])
f1, p, r, per, val_truth, val_pred = evaluate_heldout(model, val_ids)
print(f"Best epoch {best['epoch']} | strict macro-F1 {f1:.4f} | P {p:.4f} | R {r:.4f}")
for l in LABELS:
    print(f"  {l:10s} P={per[l][0]:.3f} R={per[l][1]:.3f} F1={per[l][2]:.3f}  (gold n={lab_counts.get(l,0)})")
# Files you can feed to the official evaluate_phase2.py if you want a cross-check:
json.dump({k: [list(x) for x in v] for k, v in val_truth.items()}, open(os.path.join(OUT_DIR, "val_truth.json"), "w"))
json.dump({k: [list(x) for x in v] for k, v in val_pred.items()}, open(os.path.join(OUT_DIR, "val_pred.json"), "w"))

# %% [12] Predict the test set (fold3) -> prediction.json -> zip
test_texts = json.load(open(hf_local(TEST_TEXT_FILE), encoding="utf-8"))
print("Test docs:", len(test_texts))
test_pred = {}
t0 = time.time()
for i, (k, text) in enumerate(test_texts.items()):
    rec = encode_doc(text)
    spans = decode_spans(text, predict_token_tags(model, rec), rec)
    test_pred[k] = [[s, e, l] for (s, e, l) in spans]
    if (i + 1) % 20 == 0:
        print(f"  predicted {i+1}/{len(test_texts)} | {(time.time()-t0)/60:.1f}m")

# Validation of the file before submitting
assert set(test_pred) == set(test_texts), "ID mismatch"
for k, spans in test_pred.items():
    n = len(test_texts[k])
    for s, e, l in spans:
        assert 0 <= s < e <= n and l in LABELS, (k, s, e, l)
print("Total predicted entities:", sum(len(v) for v in test_pred.values()))
ex_k = next(k for k, v in test_pred.items() if v)
print("Sample:", [(s, e, l, test_texts[ex_k][s:e]) for s, e, l in test_pred[ex_k][:5]])

with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(test_pred, f)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_PATH, arcname="prediction.json")
print("Codabench submission ready:", ZIP_PATH)

Best epoch 4 | strict macro-F1 0.2701 | P 0.2374 | R 0.3264
  APP        P=0.055 R=0.115 F1=0.074  (gold n=407)
  RESP       P=0.040 R=0.121 F1=0.060  (gold n=381)
  A.COUNSEL  P=0.042 R=0.057 F1=0.048  (gold n=195)
  R.COUNSEL  P=0.161 R=0.377 F1=0.226  (gold n=184)
  JUDGE      P=0.209 R=0.375 F1=0.269  (gold n=245)
  WIT        P=0.000 R=0.000 F1=0.000  (gold n=187)
  AUTH       P=0.153 R=0.143 F1=0.148  (gold n=1003)
  COURT      P=0.703 R=0.912 F1=0.794  (gold n=1013)
  STAT       P=0.369 R=0.453 F1=0.407  (gold n=1260)
  PREC       P=0.170 R=0.257 F1=0.205  (gold n=1183)
  DATE       P=0.800 R=0.852 F1=0.825  (gold n=1575)
  CASENO     P=0.146 R=0.254 F1=0.185  (gold n=760)


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Test docs: 35
  predicted 20/35 | 0.1m
Total predicted entities: 5602
Sample: [(26, 40, 'COURT', 'COURT OF INDIA'), (70, 93, 'CASENO', 'CIVIL APPEAL NO.6580 OF'), (99, 100, 'CASENO', 'A'), (128, 162, 'CASENO', 'PETITION (CIVIL) NO. 15449 OF 2004'), (170, 182, 'APP', 'ABS PVT. LTD')]
Codabench submission ready: /kaggle/working/cs781_phase2_exp1/submission_phase2_exp1.zip
